In [10]:
!pip install qiskit qiskit-ibm-runtime matplotlib numpy -q


In [12]:
from qiskit_ibm_runtime import QiskitRuntimeService
import os
import numpy as np
import matplotlib.pyplot as plt

# ── RECOVER RESULTS ───────────────────────────────────────────
service = QiskitRuntimeService(
    channel="ibm_quantum_platform",
    # Set IBM_QUANTUM_TOKEN in your environment before rerunning this notebook.
    token=os.getenv("IBM_QUANTUM_TOKEN"),
    instance="crn:v1:bluemix:public:quantum-computing:us-east:a/621ce660d78e42df88dafab77bd3ed6f:ecc6fe26-1b5c-4878-8696-078250580b07::"
)

job    = service.job("d786n93c6das739hh5m0")
result = job.result()

# Fix: try all possible register names
data = result[0].data
for name in ['meas', 'c', 'cr', 'm']:
    if hasattr(data, name):
        counts = getattr(data, name).get_counts()
        print(f"✓ Got counts via register '{name}'")
        break

# ── RESULTS ───────────────────────────────────────────────────
n      = 3
target = "101"
iters  = 2
SHOTS  = 4096

p_classical   = 1 / 2**n
p_ideal       = np.sin((2*iters+1) * np.arcsin(1/np.sqrt(2**n)))**2
p_hardware    = counts.get(target, 0) / SHOTS
noise_penalty = (p_ideal - p_hardware) / p_ideal * 100

print(f"\n{'═'*50}")
print(f"  IBM Heron r2 — REAL QUANTUM HARDWARE RESULTS")
print(f"  Job: d786n93c6das739hh5m0")
print(f"{'═'*50}")
print(f"  Classical random : {p_classical:.4f}")
print(f"  Ideal Grover     : {p_ideal:.4f}")
print(f"  Real hardware    : {p_hardware:.4f}  ← measured tonight")
print(f"  Noise penalty    : {noise_penalty:.1f}%")
print(f"  Fidelity         : {100-noise_penalty:.1f}%")
print(f"  Speedup          : {p_hardware/p_classical:.2f}x over classical")
print(f"  All counts       : {dict(sorted(counts.items(), key=lambda x:-x[1]))}")
print(f"{'═'*50}")

# ── PLOT ──────────────────────────────────────────────────────
states = [format(i, f'0{n}b') for i in range(2**n)]
probs  = [counts.get(s,0)/SHOTS for s in states]
colors = ['#E85D24' if s==target else '#185FA5' for s in states]

fig, axes = plt.subplots(1, 2, figsize=(13,5))
fig.patch.set_facecolor('#0f0f0f')
fig.suptitle('PQC Research — IBM Heron r2 — Real Quantum Hardware — April 2026', color='white')

for ax in axes:
    ax.set_facecolor('#1a1a1a')
    ax.tick_params(colors='white')
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['bottom','left']].set_color('#444')

axes[0].bar(states, probs, color=colors, edgecolor='#333')
axes[0].axhline(p_classical, color='#aaa',    ls='--', lw=1, label=f'Classical {p_classical:.3f}')
axes[0].axhline(p_ideal,     color='#4CAF50', ls='--', lw=1, label=f'Ideal {p_ideal:.3f}')
axes[0].axhline(p_hardware,  color='#E85D24', ls='-',  lw=1, label=f'Hardware {p_hardware:.3f}')
axes[0].set_xlabel('State', color='white')
axes[0].set_ylabel('Probability', color='white')
axes[0].set_title(f'Grover |{target}⟩ — noise penalty {noise_penalty:.1f}%', color='white')
axes[0].legend(facecolor='#2a2a2a', edgecolor='#444', labelcolor='white', fontsize=9)

qr  = np.arange(3, 1500, 5)
eps = max(1e-9, 1-(max(p_hardware,1e-9)/p_ideal)**(1/139))
fid = np.array([(1-eps)**(q*2) for q in qr])

axes[1].semilogy(qr, fid, color='#185FA5', lw=2, label='Extrapolated fidelity')
axes[1].fill_between(qr, fid*0.2, np.minimum(fid*5,1), alpha=0.1, color='#185FA5')
axes[1].axhline(0.01, color='#E85D24', ls='--', lw=1.5, label='1% min usable')
axes[1].axvline(1200, color='#FFD700', ls='--', lw=1.5, label='1200q ECDSA threshold')
axes[1].axvline(n,    color='#4CAF50', ls=':',  lw=1.5, label=f'This run ({n}q)')
axes[1].set_xlabel('Qubit count', color='white')
axes[1].set_ylabel('Fidelity (log)', color='white')
axes[1].set_title('Why hardware cannot break ECDSA yet', color='white')
axes[1].legend(facecolor='#2a2a2a', edgecolor='#444', labelcolor='white', fontsize=8)
axes[1].set_xlim(0, 1400)
axes[1].set_ylim(1e-8, 1.5)

plt.tight_layout()
plt.savefig('grover_results.png', dpi=150, bbox_inches='tight', facecolor='#0f0f0f')
plt.show()
print("✓ Saved: grover_results.png")

qiskit_runtime_service._discover_account:WARNING:2026-04-04 01:42:03,700: Loading account with the given token. A saved account will not be used.


✓ Got counts via register 'c'

══════════════════════════════════════════════════
  IBM Heron r2 — REAL QUANTUM HARDWARE RESULTS
  Job: d786n93c6das739hh5m0
══════════════════════════════════════════════════
  Classical random : 0.1250
  Ideal Grover     : 0.9453
  Real hardware    : 0.7461  ← measured tonight
  Noise penalty    : 21.1%
  Fidelity         : 78.9%
  Speedup          : 5.97x over classical
  All counts       : {'101': 3056, '100': 175, '000': 170, '001': 154, '111': 138, '010': 138, '011': 137, '110': 128}
══════════════════════════════════════════════════
✓ Saved: grover_results.png


In [14]:
from qiskit_ibm_runtime import QiskitRuntimeService
import os

service = QiskitRuntimeService(
    channel='ibm_quantum_platform',
    instance='crn:v1:bluemix:public:quantum-computing:us-east:a/621ce660d78e42df88dafab77bd3ed6f:ecc6fe26-1b5c-4878-8696-078250580b07::'
)
job = service.job('d786n93c6das739hh5m0')
job_result = job.result()

# To get counts for a particular pub result, use
#
# pub_result = job_result[<idx>].data.<classical register>.get_counts()
#
# where <idx> is the index of the pub and <classical register> is the name of the classical register.
# You can use circuit.cregs to find the name of the classical registers.

In [15]:
from google.colab import drive, files
drive.mount('/content/drive')

import shutil, json
from datetime import datetime

# Save all outputs to Drive
folder = f'/content/drive/MyDrive/pqc-blockchain-attack-windows'
import os
os.makedirs(folder, exist_ok=True)

# Copy plot
shutil.copy('grover_results.png', f'{folder}/grover_results_ibm_heron_r2.png')

# Save all results as a research record
record = {
    "date": "2026-04-03",
    "project": "pqc-blockchain-attack-windows",
    "backend": "ibm_fez",
    "processor": "IBM Heron r2",
    "qubits_available": 156,
    "job_id": "d786n93c6das739hh5m0",
    "n_qubits": 3,
    "target_state": "101",
    "shots": 4096,
    "transpiled_depth": 139,
    "results": {
        "classical_baseline": 0.1250,
        "ideal_grover": 0.9453,
        "real_hardware": 0.7461,
        "noise_penalty_pct": 21.1,
        "fidelity_pct": 78.9,
        "quantum_speedup": 5.97,
        "counts": {"101":3056,"100":175,"000":170,"001":154,"111":138,"010":138,"011":137,"110":128}
    },
    "derived": {
        "epsilon_gate": 0.00167,
        "notes": "ε_gate = 1 - 0.789^(1/139). Used as input to Aristotle FALCON proof breakdown analysis."
    },
    "aristotle_jobs": {
        "job_1": "FALCON security proof edge case search - RUNNING",
        "job_2": "IBM Heron r2 noise vs FALCON proof breakdown - QUEUED"
    },
    "significance": "First empirical IBM Heron r2 measurement used to bound ECDSA quantum attack timeline. Input to formal Lean verification of FALCON-512 security proof under real hardware noise."
}

with open(f'{folder}/research_record.json', 'w') as f:
    json.dump(record, f, indent=2)

print("✓ Saved to Google Drive:")
print(f"  {folder}/grover_results_ibm_heron_r2.png")
print(f"  {folder}/research_record.json")
print("\nSafe to close Colab now.")

Mounted at /content/drive
✓ Saved to Google Drive:
  /content/drive/MyDrive/pqc-blockchain-attack-windows/grover_results_ibm_heron_r2.png
  /content/drive/MyDrive/pqc-blockchain-attack-windows/research_record.json

Safe to close Colab now.
